# Encontro 2 · O log da Aurora Imóveis e o coletor do colega

**Coleta e Preparação de Dados · PUCRS Online** — prática da Aula 2 (expressões regulares e APIs REST)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucaskup/pucrs-coleta-e-preparacao/blob/main/aula2/pratica/encontro2.ipynb)

A imobiliária fictícia **Aurora Imóveis** publicou uma API e está recebendo acessos de vários robôs.
Hoje você vai trabalhar dos dois lados:

| Parte | Formato | Tempo |
| --- | --- | --- |
| **A. Investigar o log** com regex | em duplas, escrevendo código | 20 min |
| **B. Revisar o coletor de um colega** | em grupo, lendo e corrigindo código | 25 min |

Salve uma cópia antes de editar: *Arquivo → Salvar uma cópia no Drive*.
Para testar padrões com destaque visual, use o
[testador de regex da disciplina](https://lucaskup.github.io/pucrs-coleta-e-preparacao/aula2/testador/).

In [ ]:
import re
from collections import Counter

import pandas as pd
import requests

BASE = "https://lucaskup.github.io/pucrs-coleta-e-preparacao"
r = requests.get(f"{BASE}/aula2/pratica/acessos.log", timeout=10)
r.raise_for_status()
log = r.text
linhas = log.splitlines()
print(len(linhas), "linhas")
print(linhas[0])

## Parte A — Investigar o log (20 min)

Cada linha segue o formato *combined* dos servidores web, com o tempo de resposta no fim:

```
IP - - [dia/Mês/ano:hh:mm:ss -0300] "MÉTODO caminho HTTP/1.1" status bytes "referer" "user-agent" rt=NNNms
```

Use `re.findall`, `re.search` e grupos de captura. Com `re.M`, `^` e `$` valem para cada linha do texto.

### A1. Quantas respostas de cada código de status?

In [ ]:
status = re.findall(r"...", log)   # complete o padrão
Counter(status).most_common()

### A2. Latência: só o número, sem `rt=` e sem `ms`
Qual a média, o máximo, e quantas requisições passaram de 3 segundos?

In [ ]:
lat = [int(x) for x in re.findall(r"...", log)]
len(lat), sum(lat) / len(lat), max(lat)

### A3. Quem foi barrado por excesso de requisições (429)? Qual é o *User-Agent* dele?

In [ ]:
barrados = re.findall(r"...", log, re.M)
Counter(barrados)

### A4. Quais robôs deixaram um e-mail de contato no *User-Agent*?

In [ ]:
contatos = set(re.findall(r"...", log))
contatos

### A5. Quem tentou acessar áreas proibidas pelo `robots.txt` (`/painel-corretor/` e `/admin/`)?

In [ ]:
intrusos = re.findall(r"...", log, re.M)
Counter(intrusos)

### A6. Desafio: transforme o log em um DataFrame
Use **um** padrão com grupos nomeados `(?P<nome>...)` e `Series.str.extract`. Depois, responda: qual endereço IP teve a maior latência média?

In [ ]:
padrao = r"..."
df = pd.Series(linhas).str.extract(padrao)
df.head()

---
## Parte B — Revise o coletor do colega (25 min)

O código abaixo **roda sem erro** e imprime um resultado. Mesmo assim, ele tem pelo menos
**sete problemas** — de correção, de robustez e de boa conduta — à luz da Parte 2 da videoaula.

1. Rode a célula e compare o resultado com o que a API declara (veja o campo `total` de qualquer página).
2. Preencha a tabela de achados.
3. Reescreva o coletor na célula seguinte.

In [ ]:
# ======== código do colega: NÃO edite esta célula ========
import requests, re, pandas as pd

TOKEN = "sk-live-9f8e7d6c5b4a3210"        # minha chave da API
base = BASE + "/aula2/api/v1"

dados = []
pagina = 1
corpo = requests.get(f"{base}/municipios/pagina-{pagina}.json",
                     headers={"Authorization": "Bearer " + TOKEN}).json()
while corpo["proxima"]:
    dados.extend(corpo["dados"])
    pagina += 1
    corpo = requests.get(f"{base}/municipios/pagina-{pagina}.json",
                         headers={"Authorization": "Bearer " + TOKEN}).json()
municipios = pd.DataFrame(dados)
print(len(municipios), "municípios coletados")

busca = requests.get(base + "/localidades/estados/RS/municipios.json?nome=" + "São José do Norte")

proc = pd.DataFrame(requests.get(base + "/processos.json").json())
proc["protocolo"] = proc["resumo"].str.extract(r"(\d{4}/\d{5})")
proc[["orgao", "protocolo"]]

### Achados

| # | Linha / trecho | Problema | Consequência | Correção |
| --- | --- | --- | --- | --- |
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |
| 6 | | | | |
| 7 | | | | |

In [ ]:
# sua versão do coletor

---
### Para a apresentação (40 min)
Cada grupo apresenta **um achado** da Parte B por rodada (sem repetir os anteriores), mostrando o trecho
do código, a consequência e a correção. Depois, a turma compara as soluções da Parte A, em especial
a regex da A6.